In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("../data/KedarkanthaKotgaonItinerary.pdf")
docs = loader.load()
len(docs)

2

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [5]:
len(splitted_docs)

5

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
  documents=splitted_docs,
  embedding=embeddings
)

#### Agent = Tools, LLM, Prompt

In [7]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve data of the PDF Documents, and these 
        documents have details about Kedarkantha Kotgaon Itinerary.
    """
    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""
    
    for doc in docs:
        context = doc.page_content + "\n\n"
        
    return context

In [8]:
llm = ChatOpenAI(model="gpt-5-nano")

In [9]:
system_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    Always use the `retriever_tool` for questions requiring external knowledge.
"""

In [10]:
agent =create_agent(
  model=llm,
  tools=[retriever_tool],
  system_prompt=system_Prompt
)

In [11]:
query = "What are the trek highlights and what is the contact number?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  Kedarkantha Kotgaon itinerary trek highlights and contact number
Tool Called:  Kedarkantha Kotgaon trek highlights


In [12]:
result = response["messages"][-1].content

In [13]:
print(result)

Here are the trek highlights and the contact number from the Kedarkantha Kotgaon itinerary:

Trek highlights
- The finest summit climb for beginners
- Highest altitude: 12,500 ft
- Duration: 6 days (from Dehradun to Dehradun)
- Base camp: Kotgaon / Gaichawan Gaon, Uttarakhand
- Accommodation: Camping
- Cloakroom facility: Available
- Fees: ₹11,450
- Age limit: 8 to 62+ years
- Fitness expectation: Ability to jog 5 km in 38 minutes
- From base camp to base camp logistics noted in the itinerary

Contact number
- 08046801269
